# Deep Learning  - Análise de Sentimentos


In [189]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Trabalha na conversão e padronização dos dados
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [190]:
dados = pd.read_csv('sentimentos_ptbr.csv')
dados.head()

,texto,sentimento
0,Eu me arrependi da compra do ventilador; ele n...,0
1,É a segunda vez que tenho problemas com este t...,0
2,Eu não me arrependi da compra do impressora; e...,1
3,Minha experiência foi ótima. O fone de ouvido ...,1
4,Minha experiência foi péssima. O impressora nã...,0


In [191]:
dados.shape

(1600, 2)

In [192]:
dados['sentimento'].value_counts()

,count
sentimento,
0,800
1,800


## Transformar os dados em vetores

In [193]:
# Transformar os dados em vetores
X = dados['texto'].to_numpy()
y = dados['sentimento'].to_numpy()

## Separar dados para treinamento e testes

In [194]:
X_desenvolvimento, X_teste, y_desenvolvimento, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Stratify -> tentar manter a proporção de dados positivos e negativos

In [195]:
# Separar um pedaço dos dados para validação. Ajudar a rede neural se está aprendendo bem sem esperar chegar no teste
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_desenvolvimento,
    y_desenvolvimento,
    test_size=0.2,
    random_state=42,
    stratify=y_desenvolvimento
)

In [196]:
print("Treino: ", len(X_treino))
print("Validacao: ", len(X_validacao))
print("Teste: ", len(X_teste))

Treino:  1024
Validacao:  256
Teste:  320


## Rede Neural

In [197]:
# Converter palavras em numeros

vetorizador = layers.TextVectorization(
    max_tokens=5000,
    output_sequence_length=100
)

# Adapt -> Aprende as palavras que existem no texto de trainamento
vetorizador.adapt(X_treino)

In [198]:
# Frase de exemplo
print(vetorizador("O produto é excelente"))

tf.Tensor(
[ 2 75  9 30  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0], shape=(100,), dtype=int64)


### Funções de ativação - Modificar a saida dos neuronios


In [199]:
valores = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

# ReLU -> troca numeros negativos por 0 e mantem os positivos
relu = np.maximum(0, valores)

# Tanh - Comprimir os valores entre -1 e 1
tanh = np.tanh(valores)

# Sigmoide - Comprimir valores entre 0 e 1
sigmoide = 1 / (1 + np.exp(-valores))

print("Entradas: ", valores)
print("ReLU: ", np.round(relu, 2))
print("Tanh: ", np.round(tanh, 2))
print("Sigmoide: ", np.round(sigmoide, 2))

Entradas:  [-2. -1.  0.  1.  2.]
ReLU:  [0. 0. 0. 1. 2.]
Tanh:  [-0.96 -0.76  0.    0.76  0.96]
Sigmoide:  [0.12 0.27 0.5  0.73 0.88]


### Construção da rede naural

In [200]:
modelo = keras.Sequential()

In [201]:
# Input define o formato de campo do texto, em string
modelo.add(layers.Input(shape=(1,), dtype="string"))

In [202]:
# Adiciona o vocabulário e converte palavras em numeros
modelo.add(vetorizador)

In [203]:
# Embeddings - Transforma cada código em um vetor
# mask_zero=True informa que os '0' dentro do vetor são palavras vazias, e não informações reais
modelo.add(layers.Embedding(input_dim=5000, output_dim=16, mask_zero=True))

In [204]:
# GlobalAvarage - Resume textos e vatores
# Atenção: pode perder muitos detlahes nas frases
modelo.add(layers.GlobalAveragePooling1D())

In [205]:
# Camada com 16 neuronios e camada de ativação Relu
modelo.add(layers.Dense(16, activation='relu'))

In [206]:
# Desativa aleatoriamente 30 das saídas do treinamento
modelo.add(layers.Dropout(0.3))

In [207]:
# Camada de saída com 1 neuronio e ativação Sigmoid
modelo.add(layers.Dense(1, activation='sigmoid'))

In [208]:
modelo.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_9            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_8 (Embedding)         │ (None, 100, 16)        │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,289 (313.63 KB)

 Trainable params: 80,289 (313.63 KB)

 Non-trainable params: 0 (0.00 B)

In [209]:
modelo.compile(optimizer='adam',
               loss='binary_crossentropy',
               metrics=['accuracy'])

In [210]:
# Se não melhorar o aprendizado em X épocas, o treinamento é interrompido
parada_antecipada = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [211]:
historico = modelo.fit(
    X_treino,
    y_treino,
    epochs=500,
    batch_size=32, # Processa as informacoes em grupo
    validation_data=(X_validacao, y_validacao), # Medir os erros dos textos e ajustar os pesos
    callbacks=[parada_antecipada] # Regra de parada antecipada
)

Epoch 1/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7646 - loss: 0.6855 - val_accuracy: 0.9727 - val_loss: 0.6717
Epoch 2/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9141 - loss: 0.6450 - val_accuracy: 0.9883 - val_loss: 0.6118
Epoch 3/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9688 - loss: 0.5623 - val_accuracy: 0.9961 - val_loss: 0.5092
Epoch 4/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9844 - loss: 0.4487 - val_accuracy: 1.0000 - val_loss: 0.3875
Epoch 5/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9932 - loss: 0.3379 - val_accuracy: 1.0000 - val_loss: 0.2758
Epoch 6/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.2375 - val_accuracy: 1.0000 - val_loss: 0.1901
Epoch 7/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9990 - loss: 0.1723 - val_accuracy: 1.0000 - val_loss: 0.1305
Epoch 8/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9990 - loss: 0.1282 - val_accuracy: 1.0000 - 

In [212]:
# Análise do resultado
perda, acuracia = modelo.evaluate(X_teste, y_teste)
print("Perda: ", perda)
print("Acuracia: ", acuracia)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 1.6484e-05
Perda:  1.6484436855535023e-05
Acuracia:  1.0


In [213]:
# Gera um score de sentimento para cada comentário
probabilidades = modelo.predict(X_teste).ravel()

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step  


In [214]:
# Transforma a regra do score em:
# <0.5 = 0 (negativo)
# >=0.5 = 1 (positivo)
previsoes = (probabilidades >=0.5).astype(int)

In [215]:
print(confusion_matrix(y_teste, previsoes))

[[160   0]
 [  0 160]]


In [216]:
print(classification_report(y_teste, previsoes, target_names=['Negativo', 'Positivo']))

              precision    recall  f1-score   support

    Negativo       1.00      1.00      1.00       160
    Positivo       1.00      1.00      1.00       160

    accuracy                           1.00       320
   macro avg       1.00      1.00      1.00       320
weighted avg       1.00      1.00      1.00       320



In [227]:
# Validação real
frase = 'O produto veio quebrado'

resultado = modelo.predict(np.array([frase], dtype=object))

score = float(resultado[0][0])
if score >= 0.5:
    print("Positivo: ", score)
else:
    print("Negativo: ", score)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Negativo:  8.577339372095594e-08
